# SFT version of 03 neweval (built by `student/make_sft_notebooks.py`, plan in `student/SFT_PLAN.md`)

`infer.py` is unchanged, so scores compare with neweval's. Changes in the transcript cell:
1. **Offline run**: with no `raw_asr*.json` attached, every clip comes from the attached 01c output. That is what the Docker
   can score (no API ASR at inference, R-RUN-1). Attach raw files only to measure another ASR.
2. **EVAL on by default**, with the held-out train clips from `asr_xfit_train.json` (attach the `thbased01_sft` output), whose
   models never heard them. Raw files still win where they have a clip.
3. **Guard**: EVAL stops when the 01c text gives a held-out WER under 0.05, which means that ASR trained on those clips
   (01c FINAL: 0.0194, stock 0.1609), so the score would be inflated.

Attach the LLM output to test and say which one in the results: neweval tested `qwenupdatefabe`, not `bowen`.


# 03 ASR compare · Qwen on another model's transcripts

`adahome/03c_eval_repair.ipynb` with one change: the ASR text comes from a raw file written by any other ASR model, and this notebook converts it the way 01c does (`norm()`, then `vi_itn.py`) before the agent sees it. `infer.py` is byte-identical to 03c repair, so with the same LLM the score change is the ASR. See `instructions.md` beside this notebook.

**Attach** (Kaggle, GPU T4 x2, internet on):
1. the data package (private dataset),
2. the LLM output to test, normally `png0806/bowen` (`llm_model/`, `lifemate.py`),
3. one private dataset holding the transcripts as `raw_asr*.json`, for example `raw_asr_public.json` and `raw_asr_train.json`. All matches are merged.

`vi_itn.py` is written by a cell below, so no dataset needs it. Attach a 01c output only with `FILL_FROM_01C = True`: clips missing from the raw files (for example history turns) then come from its `asr_transcripts.json`.

**Raw file format**: one JSON object `{key: text}`, key = `<split>/<audio path from <split>.jsonl>`, for example `"public_test/audio/public_0002/public_0002.wav"`. One key per final request and per user turn in `history` (1,933 for public_test). Text can have capitals, punctuation and digits. A failed clip is `""`. The Gemini cache `gemini_asr_public_test.json` already has this format: rename it to `raw_asr_public.json`.

**Settings**: `EVAL = True` needs the 1,089 clips of the 328 held-out train samples in the raw files, and the cell stops if any is missing. Set it `False` if you transcribed public_test only. `GOLD = False` skips the gold-text run (known: 71.77 for `bowen`).

**Rules**: these transcripts are for measurement only. Never train or tune on public_test transcripts (labelling test data). No API ASR except Gemini under rule 6's second exception, and never in the Docker.

**Output**: `asr_transcripts.json` (converted), `pred_*.jsonl`, `submission.jsonl`, held-out WER and score next to the `bowen` + 01c baseline (public 60.29).


In [ ]:
import glob
import json

import torch
assert torch.cuda.device_count() == 2, "Set Settings > Accelerator to GPU T4 x2"
print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(2)])
_config = json.load(open(glob.glob("/kaggle/input/**/llm_model/config.json", recursive=True)[0], encoding="utf-8"))
TRANSFORMERS = {"spark2_5": "4.57.1", "lfm3": "5.20.2"}.get(_config["model_type"], "5.22.1")  # the version each model was trained on
print(_config["model_type"], "transformers", TRANSFORMERS)
!pip install -q "transformers=={TRANSFORMERS}" "accelerate==1.15.0" "jiwer==4.0.0"


In [ ]:
%%writefile settings.py

import glob
from pathlib import Path

SMOKE = False  # True: 8 samples per GPU
EVAL = True  # held-out train rows, clips from the raw files, else asr_xfit_train.json, else 01c (only if FINAL = False). False: public_test only
FILL_FROM_01C = True  # also on by itself when an asr_transcripts.json is attached. True: clips missing from raw_asr*.json come from an attached 01c asr_transcripts.json (e.g. history turns), so only the rest is the new ASR
GOLD = False  # also run gold text on the held-out samples (bowen: 71.77, already known)
DTYPE = "float16"
SHARDS = 2  # one process per T4
MAX_STEPS = 8  # gold never uses more than 6 calls
MAX_NEW_TOKENS = 320  # the longest gold call is 304 characters, plus a written reason when the model was trained with WHY
TIME_BUDGET = 120.0  # a file upload is not timed. The Docker must stay under the real limit of 30 s: 25 shows what that limit costs
STRIP_MARKS = True  # remove the tagger's [ ] from ASR text. False: the old behaviour
SEED = 42

ROOT = Path(glob.glob("/kaggle/input/**/train/train.jsonl", recursive=True)[0]).parents[1]
LLM = str(Path(glob.glob("/kaggle/input/**/llm_model/config.json", recursive=True)[0]).parent)
LIFEMATE_DIR = str(Path(LLM).parent)  # lifemate.py as saved with these weights
RAW_JSON = sorted(glob.glob("/kaggle/input/**/raw_asr*.json", recursive=True))  # the other ASR model's output, merged. None: offline run on the 01c output only
ASR_JSON = "asr_transcripts.json"  # written by the next cell, in 01c's style

In [ ]:
%%writefile vi_itn.py
"""Inverse text normalisation for lowercase Vietnamese ASR output: spoken numbers become digits, everything else stays.
"chuyển hai trăm nghìn lúc chín rưỡi" -> "chuyển 200000 lúc 9 rưỡi". Text inside [entity marks] is left alone.
Rules come from train transcripts only. Self-check: python vi_itn.py"""
import re

DIGIT = {"không": 0, "một": 1, "hai": 2, "ba": 3, "bốn": 4, "năm": 5, "sáu": 6, "bảy": 7, "tám": 8, "chín": 9}
ONES = DIGIT | {"mốt": 1, "tư": 4, "lăm": 5}  # the last three only mean a number after another number
SCALE = {"tỷ": 10**9, "tỉ": 10**9, "triệu": 10**6, "nghìn": 1000, "ngàn": 1000}
BEFORE = {"tháng", "mùng", "mồng", "tầng", "lầu", "quận", "kênh", "tập", "mức", "kém", "số", "ngày"}  # a lone digit after these is a number
UNITS = {"giờ", "tiếng", "phút", "giây", "độ", "phần", "chỗ", "vé", "mét", "ký", "kg", "km", "lít", "đô", "đồng", "rưỡi", "ngày", "tuần", "tháng"}  # and before these
YEAR = {"nay", "sau", "ngoái", "trước"}  # "năm" as year


def at(w, i):
    return w[i] if i < len(w) else ""


def value(a):
    return DIGIT[a] if a in DIGIT else int(a) if a.isdigit() else None


def tens(w, i):
    """mười lăm, hai mươi mốt, năm chục, and the spoken short forms hai mốt, bốn lăm, hai sáu độ, hôm hai tám."""
    a, b = at(w, i), at(w, i + 1)
    if a == "mười":
        v, j = 10, i + 1
    elif DIGIT.get(a) and b in ("mươi", "chục"):
        v, j = 10 * DIGIT[a], i + 2
    elif DIGIT.get(a, 0) >= 2 and (b in ("mốt", "tư", "lăm") or (b in DIGIT and (at(w, i + 2) in ("độ", "phần") or (i and w[i - 1] in ("ngày", "hôm", "mùng", "mồng"))))):
        return 10 * DIGIT[a] + ONES[b], i + 2
    else:
        return None
    c = at(w, j)  # "năm" here is a year: mười năm, hai mươi năm. Five after a ten is spoken "lăm".
    return (v + ONES[c], j + 1) if c in ONES and c not in ("không", "năm") else (v, j)


def tail(w, j, scale):
    """hai triệu tám, hai triệu rưỡi: the unit below the scale is implied. "một" after a number means per: mười tám nghìn một ký."""
    a = at(w, j)
    if a == "rưỡi":
        return scale // 2, j + 1
    if a != "một" and ONES.get(a) and at(w, j + 1) not in UNITS | YEAR:
        return ONES[a] * scale // 10, j + 1
    return 0, j


def group(w, i):
    """One number below 1000: (value, next index), or None."""
    a, b = at(w, i), at(w, i + 1)
    if value(a) is not None and b == "trăm":
        v, j = 100 * value(a), i + 2
    elif a == "trăm" and (tens(w, i + 1) or DIGIT.get(b, 0) >= 2):  # trăm hai mốt
        v, j = 100, i + 1
    else:
        return tens(w, i) or ((value(a), i + 1) if value(a) is not None and b in SCALE else None)
    if at(w, j) in ("lẻ", "linh") and at(w, j + 1) in DIGIT:
        return v + DIGIT[w[j + 1]], j + 2
    t = tens(w, j)
    if t:
        return v + t[0], t[1]
    r, j = tail(w, j, 100)
    return v + r, j


def cardinal(w, i):
    """Longest number at w[i]: (value, next index), or None."""
    total, j, scale = 0, i, 0
    while g := group(w, j):
        v, k = g
        s = SCALE.get(at(w, k), 0)
        if not s or (scale and s >= scale):
            return (total + v, k) if not s else (total, j)
        total, j, scale = total + v * s, k + 1, s
    if not scale:
        return None
    r, j = tail(w, j, scale)
    return total + r, j


def digits(s):
    w, out, i = s.split(), [], 0
    while i < len(w):
        n = i
        while at(w, n) in DIGIT:
            n += 1
        n -= n - i > 3 and at(w, n) in UNITS  # rv hai một chín tám giờ: the last digit belongs to the unit
        if i and w[i - 1] == "thứ":  # thứ hai is Monday
            out.append(w[i])
            i += 1
        elif c := cardinal(w, i):
            out.append(str(c[0]))
            i = c[1]
        elif n - i >= 3:  # ponytail: any three digit words in a row become a digit string, "ba năm một lần" would too
            out.append("".join(str(DIGIT[x]) for x in w[i:n]))
            i = n
        elif w[i] in DIGIT and w[i] != "không" and (at(w, i - 1) in BEFORE and i or at(w, i + 1) in UNITS):
            out.append(str(DIGIT[w[i]]))
            i += 1
        else:
            out.append(w[i])
            i += 1
    return " ".join(out)


def itn(s):
    return " ".join(p if p[0] == "[" else digits(p) for p in re.split(r"(\[[^\]]*\])", s) if p.strip())


if __name__ == "__main__":
    for a, b in {
        "chuyển lan hai trăm nghìn": "chuyển lan 200000",
        "hai triệu ba trăm tám chục nghìn": "2380000",
        "một triệu rưỡi tiền cọc": "1500000 tiền cọc",
        "hai triệu tám và nạp": "2800000 và nạp",
        "hai triệu tám giờ sáng": "2000000 8 giờ sáng",
        "mười tám nghìn một ký": "18000 1 ký",
        "trăm hai mốt triệu": "121000000",
        "bốn chục triệu": "40000000",
        "năm trăm đô": "500 đô",
        "bật điều hoà hai sáu độ": "bật điều hoà 26 độ",
        "số không chín một hai bốn năm sáu bảy tám một ba": "số 09124567813",
        "chín giờ kém mười lăm": "9 giờ kém 15",
        "năm giờ bốn lăm": "5 giờ 45",
        "đặt báo thức sáu rưỡi sáng mai": "đặt báo thức 6 rưỡi sáng mai",
        "thứ hai tuần sau bay một mình": "thứ hai tuần sau bay một mình",
        "ngày mốt mở một bài": "ngày mốt mở một bài",
        "có không ba ơi năm nay": "có không ba ơi năm nay",
        "mười năm rồi": "10 năm rồi",
        "tháng chín mùng bốn": "tháng 9 mùng 4",
        "gọi [anh ba] năm phút": "gọi [anh ba] 5 phút",
        "[số một] hai mươi": "[số một] 20",
        "vài trăm nghìn": "vài trăm nghìn",
        "119 triệu": "119000000",
        "rv hai một chín tám giờ": "rv 219 8 giờ",
        "lịch hôm hai tám": "lịch hôm 28",
        "": "",
    }.items():
        assert itn(a) == b, (a, itn(a), b)
    print("vi_itn self-check ok")


In [ ]:
import glob
import json
import re
import sys
import unicodedata

import jiwer

from settings import *

sys.path[:0] = [".", LIFEMATE_DIR]  # vi_itn.py is written by the cell above
from vi_itn import itn
import lifemate as lm

TONES = {"\u0300", "\u0301", "\u0303", "\u0309", "\u0323"}


def old_style_tone(w):
    """Same as 01c: hoà -> hòa, khoẻ -> khỏe, thuỷ -> thủy."""
    d = unicodedata.normalize("NFD", w)
    t = [c for c in d if c in TONES]
    if len(t) != 1:
        return w
    b = unicodedata.normalize("NFC", "".join(c for c in d if c not in TONES))
    if b.endswith(("oa", "oe")) or (b.endswith("uy") and not b.endswith("quy")):
        return unicodedata.normalize("NFC", b[:-2] + b[-2] + t[0] + b[-1])
    return w


def norm(s):
    """01c's norm(), plus thousands separators joined first (200.000đ -> 200000đ), which Gipformer never writes."""
    s = re.sub(r"(?<=\d)[.,](?=\d{3}(?!\d))", "", unicodedata.normalize("NFC", s))
    s = re.sub(r"[^\w\s]|_", " ", s.lower())
    return " ".join(old_style_tone(w) for w in s.split())


raw = {}
for p in RAW_JSON:
    part = json.load(open(p, encoding="utf-8"))
    print(p, len(part), "clips")
    raw |= part
new = set(raw)
base = glob.glob("/kaggle/input/**/asr_transcripts.json", recursive=True)
print("01c asr_transcripts.json attached:", base)
assert RAW_JSON or base, "attach raw_asr*.json, or a 01c output for an offline run"
print("offline run: every clip from 01c" if not RAW_JSON else f"{len(new)} clips from the raw files")
if FILL_FROM_01C or base:  # 01c text is already norm() + itn(), and both are idempotent on it
    assert len(base) == 1, f"FILL_FROM_01C needs exactly one attached asr_transcripts.json, found {base}"
    raw = json.load(open(base[0], encoding="utf-8")) | raw
xfit = glob.glob("/kaggle/input/**/asr_xfit_train.json", recursive=True)
print("cross-fitted train transcripts:", xfit)
if xfit:  # from fold models that never heard these clips, so the held-out score is not inflated. The raw files still win
    raw |= {f"train/{k}": v for k, v in json.load(open(xfit[0], encoding="utf-8")).items() if f"train/{k}" not in new}
need = {s: [f"{s}/{c}" for x in rows for c in lm.clips(x)] for s, rows in
        (("public_test", lm.load_split(ROOT, "public_test")), ("train", sum(lm.split_rows(lm.load_split(ROOT, "train"))[1:], [])))}
for s, keys in need.items():
    miss = [k for k in keys if k not in raw]
    print(f"{s}: {len(keys) - len(miss)} of {len(keys)} clips present, {sum(k in new for k in keys)} from the new ASR, {sum(not raw.get(k, 'x').strip() for k in keys)} empty")
    assert not miss or (s == "train" and not EVAL), f"{len(miss)} {s} clips missing, e.g. {miss[:3]}" + (". Set EVAL = False for public_test only" if s == "train" else ". Attach the 01c output (thbased01, the one with asr_transcripts.json) to fill them")
asr = {k: itn(norm(v)) for k, v in raw.items()}
json.dump(asr, open(ASR_JSON, "w", encoding="utf-8"), ensure_ascii=False)

if EVAL:  # same WER as 01c reports after ITN: both sides through norm() and itn()
    truth = {f"train/{c}": t for x in sum(lm.split_rows(lm.load_split(ROOT, "train"))[1:], []) for c, t in lm.gold_texts(x).items()}
    keys = sorted(truth)
    w = jiwer.wer([itn(norm(truth[k])) for k in keys], [asr[k] for k in keys])
    print(f"held-out WER {w:.4f} on {len(keys)} clips (01b stock 0.1609, 01c FINAL heard them: 0.0194)")
    assert w > 0.05 or any(k in new for k in keys), "this ASR trained on the held-out clips, the score would be inflated: attach asr_xfit_train.json or a FINAL = False 01c output"
for k in need["public_test"][:5]:
    print(repr(raw[k][:120]), "->", repr(asr[k][:120]))


In [ ]:
%%writefile infer.py
"""One shard of the agent: python infer.py <split> <gold|asr> <shard>. One process per split, each split has its own zaic_tools."""
import json
import re
import sys
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from transformers.integrations import sdpa_attention

from settings import *

sdpa_attention.use_gqa_in_sdpa = lambda *args, **kwargs: False  # a T4 has no grouped-key kernel: Qwen3.5 fell back to float32 math and ran out of memory on long prompts (public_0110, public_0157)

sys.path[:0] = [LIFEMATE_DIR, str(ROOT / sys.argv[1])]
import lifemate as lm
from zaic_tools import ToolEnv

split, mode, shard = sys.argv[1], sys.argv[2], int(sys.argv[3])
set_seed(SEED)
tok = AutoTokenizer.from_pretrained(LLM)
model = AutoModelForCausalLM.from_pretrained(LLM, dtype=DTYPE, device_map="auto" if torch.cuda.is_available() else None, trust_remote_code=True).eval()
stop_id = tok.convert_tokens_to_ids(lm.STOP)  # not every repo has a generation_config, so the default eos may never fire
catalog = lm.load_catalog(ROOT, split)
rows = lm.load_split(ROOT, split)
asr = json.load(open(ASR_JSON, encoding="utf-8")) if mode == "asr" else {}
if split == "train":
    rows = sum(lm.split_rows(rows)[1:], [])  # only held-out samples say anything
rows = rows[shard::SHARDS][: 8 if SMOKE else None]


@torch.inference_mode()
def generate(msgs):
    ids = tok(lm.prompt_text(msgs), return_tensors="pt", add_special_tokens=False).to(model.device)
    assert ids.input_ids.shape[1] + MAX_NEW_TOKENS <= 16384, "over the competition context limit"
    out = model.generate(**ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, eos_token_id=stop_id, pad_token_id=stop_id)
    return tok.decode(out[0, ids.input_ids.shape[1]:], skip_special_tokens=True)


def clean(text):
    return re.sub(r"[\[\]]", "", text) if STRIP_MARKS else text  # norm() removes all punctuation, so every bracket here is a mark


def missing(name, args, installed):
    """The ask gold would make for an installed tool that misses a required or one-of parameter, else None. Reads too: gold asks for them 7 times (flights.search date). A one-of group names its first member, as gold does for ewallet.transfer (contact_id)."""
    tool = catalog.get(name)
    if tool is None or name.split(".")[0] not in installed:
        return None
    p = tool.get("parameters", {})
    empty = lambda k: args.get(k) in (None, "", [], {})  # zaic_tools counts these as missing
    need = [k for k in p.get("required", []) if empty(k)]
    if p.get("x-one-of") and all(any(empty(k) for k in grp) for grp in p["x-one-of"]):
        need.append(p["x-one-of"][0][0])
    return {"tool_name": name, "params": need} if need else None


def run(item):
    env = ToolEnv(split, item["id"], root=ROOT)  # exactly one per sample, every call on it goes into the submission
    texts = lm.gold_texts(item) if mode == "gold" else {c: clean(asr[f"{split}/{c}"]) for c in lm.clips(item)}
    msgs = lm.build_messages(item, texts, catalog)
    t0, invalid, failed, stop, raw = time.time(), 0, 0, "max_steps", []
    retried = False
    try:
        for _ in range(MAX_STEPS):
            if time.time() - t0 > TIME_BUDGET:
                stop = "budget"
                break
            raw.append(generate(msgs))
            action = lm.parse_action(raw[-1])
            if action is None:  # parse_action returns None for both, and it stays as saved with the weights
                stop = "done" if '"done"' in raw[-1] else "unparsable"
                break
            call = {"name": action[0], "arguments": action[1]}
            said = json.JSONDecoder().raw_decode(raw[-1][raw[-1].index("{"):])[0]  # the model's own JSON, so a trained "why" stays in context. Equals call for other models
            check = env.validate(*action)
            if not check["ok"]:  # validate is not logged: a call we never make costs no U penalty
                ask = missing(*action, item["context"]["installed_apps"])
                if ask and env.validate(lm.ASK, ask)["ok"]:  # gold ends on its ask too
                    env.call(lm.ASK, ask)
                    stop = "repaired"
                    break
                if not retried:  # one more try with the error as the result (4 of 5 invalid stops on public called an uninstalled contacts.search)
                    retried = True
                    msgs += [lm.asst(said), lm.result_msg(check)]
                    continue
                invalid, stop = 1, "invalid"
                break
            if env.get_tool_calls().count(call) >= 2:  # greedy loop, gold repeats a call at most once
                stop = "repeat"
                break
            if catalog.get(call["name"], {}).get("kind") == "read" and [c["name"] for c in env.get_tool_calls()].count(call["name"]) >= 2:  # gold reads one tool 3+ times in 1 of 1,545 samples; LFM3 ver3 retried empty searches 127 times on public, then sometimes wrote on a guess
                stop = "repeat"
                break
            result = env.call(*action)
            msgs += [lm.asst(said), lm.result_msg(result)]
            if "query" in action[1] and not catalog[call["name"]]["parameters"].get("required") and any(v == [] for v in result.values()):  # gold never reads an empty list (0 of 234 contacts.search): the query was misheard, so list everything once and let the model pick. 25 public samples hit this (contacts, event_tickets); a full list fits MAX_RESULT_CHARS
                msgs += [lm.asst({"name": call["name"], "arguments": {}}), lm.result_msg(env.call(call["name"], {}))]
            if call["name"] == lm.ASK:  # gold always ends here
                stop = "ask"
                break
    except Exception as e:  # keep the calls already made, a crash must not lose the whole line
        failed, stop = 1, "crash"
        print(item["id"], repr(e), flush=True)
    return {"id": item["id"], "tool_calls": env.get_tool_calls(), "state_hash": env.get_state_hash(), "sec": time.time() - t0, "invalid": invalid, "failed": failed, "stop": stop, "raw": raw}


with open(f"pred_{split}_{mode}_{shard}.jsonl", "w", encoding="utf-8") as f:
    for n, item in enumerate(rows):
        line = run(item)
        if split == "train":  # scoring only: the second env replays gold for its hash, nothing here reaches a submission
            gold = item["label"]["tool_calls"]
            ref = ToolEnv(split, item["id"], root=ROOT)
            for c in gold:
                ref.call(c["name"], c["arguments"])
            line["score"] = lm.score(line["tool_calls"], gold, line["state_hash"], ref.get_state_hash(), catalog)
            line["unseen"] = bool(lm.used_apps(item) & lm.HOLDOUT_APPS)
        f.write(json.dumps(line, ensure_ascii=False) + "\n")
        f.flush()
        if n % 25 == 0:
            print(split, mode, shard, n, "/", len(rows), f"{line['sec']:.1f}s", flush=True)

In [ ]:
from settings import *

print("LLM:", LLM, "\nASR:", RAW_JSON, "->", ASR_JSON)

if EVAL:
    if GOLD:
        !CUDA_VISIBLE_DEVICES=0 python infer.py train gold 0 & CUDA_VISIBLE_DEVICES=1 python infer.py train gold 1 & wait
    !CUDA_VISIBLE_DEVICES=0 python infer.py train asr 0 & CUDA_VISIBLE_DEVICES=1 python infer.py train asr 1 & wait
!CUDA_VISIBLE_DEVICES=0 python infer.py public_test asr 0 & CUDA_VISIBLE_DEVICES=1 python infer.py public_test asr 1 & wait


In [ ]:
import collections
import json
import statistics as st


def load(split, mode):
    return [json.loads(l) for f in sorted(glob.glob(f"pred_{split}_{mode}_*.jsonl")) for l in open(f, encoding="utf-8")]


def report(name, part):
    if part:
        s = sorted(x["sec"] for x in part)
        print(f"{name:20s} n={len(part):4d} score={100 * st.mean(x['score'] for x in part):6.2f} invalid={sum(x['invalid'] for x in part)} failed={sum(x['failed'] for x in part)} p95={s[int(.95 * len(s))]:.1f}s max={s[-1]:.1f}s")


if EVAL:
    for mode in ("gold", "asr"):
        p = load("train", mode)
        report(f"{mode} unseen", [x for x in p if x["unseen"]])
        report(f"{mode} seen", [x for x in p if not x["unseen"]])
        report(f"{mode} under 25 s", [x for x in p if x["sec"] <= 25])
        report(f"{mode} over 25 s", [x for x in p if x["sec"] > 25])
        for stop in sorted({x["stop"] for x in p}):
            report(f"{mode} {stop}", [x for x in p if x["stop"] == stop])  # repaired: was invalid in 03c

In [ ]:
pred = load("public_test", "asr")
assert len({x["id"] for x in pred}) == len(pred)
expected = sum(1 for _ in open(ROOT / "public_test/public_test.jsonl", encoding="utf-8"))
assert SMOKE or len(pred) == expected, f"{len(pred)} lines, expected {expected}"
with open("submission.jsonl", "w", encoding="utf-8") as f:
    for x in pred:
        f.write(json.dumps({k: x[k] for k in ("id", "tool_calls", "state_hash")}, ensure_ascii=False) + "\n")
print(collections.Counter(x["stop"] for x in pred).most_common(), sum(x["sec"] > 30 for x in pred), "samples over 30 s")
for x in [x for x in pred if x["stop"] == "unparsable"][:5]:
    print(x["id"], repr(x["raw"][-1][:200]))
print(f"{len(pred)} lines, {sum(x['failed'] for x in pred)} failed, {sum(x['invalid'] for x in pred)} stopped on an invalid call, slowest {max(x['sec'] for x in pred):.1f}s")
!cd "{ROOT}/public_test" && PYTHONPATH=. python -m zaic_tools --split public_test --root .. --validate /kaggle/working/submission.jsonl
print("Baseline, bowen + 01c thbased01: public 60.29, held-out gold 71.77, held-out asr 66.06 (inflated, 01c heard those clips)")
